# 3. Behavioral segments and anomaly candidates
Compare candidate K-Means sizes, choose a stable/interpretable solution, and rank unusual customer behavior.

In [ ]:
import sys
from pathlib import Path
import plotly.express as px
sys.path.insert(0, str(Path.cwd().parents[1] / 'src'))
from transaction_analysis import choose_k, segment_customers, detect_customer_anomalies
features = __import__('polars').read_parquet('../customer_features.parquet')
feature_columns = ['transaction_count','gross_amount','debit_share','credit_share','transaction_type_count','destination_bank_count','recency_days','transactions_per_active_day']
scores = choose_k(features, feature_columns)
px.line(scores.to_pandas(), x='k', y=['inertia','silhouette'], title='Candidate segment sizes').show()
scores

In [ ]:
N_CLUSTERS = 3  # choose after reviewing silhouette, size, stability, and business meaning
assignments, metadata = segment_customers(features, feature_columns, N_CLUSTERS)
assignments.write_parquet('../customer_segments.parquet')
metadata['profiles']

In [ ]:
anomalies = detect_customer_anomalies(features, feature_columns, contamination=0.02)
anomalies.filter(__import__('polars').col('anomaly_label') == -1).head(50).write_parquet('../customer_anomalies.parquet')
anomalies.head(20)

Segments are descriptive. Anomaly candidates require transaction-level drill-down and human review; do not label them fraudulent from this model alone.